# Spatial Queries

## Overview
Spatial queries combine geometry predicates with standard SQL to answer location-based questions: which sites are within this watershed? Which observations are near this protected area? What is the nearest monitoring site to each species observation?

**Key spatial JOIN patterns:**
```sql
-- Point in polygon (most common):
SELECT s.*, w.name AS watershed
FROM   monitoring_sites s
JOIN   watersheds w ON ST_Within(s.geom, w.geom)

-- Within distance (uses spatial index):
SELECT * FROM monitoring_sites
WHERE ST_DWithin(geom, ST_MakePoint(-66.06, 45.95)::geography, 50000)  -- 50km

-- Nearest neighbour (KNN — fast with LATERAL + <->):
SELECT s.site_id, s.site_name, o.geom <-> s.geom AS dist
FROM observations o
CROSS JOIN LATERAL (
    SELECT * FROM monitoring_sites ORDER BY geom <-> o.geom LIMIT 1
) s
```

---

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import pyproj

# ── Ecological dataset: Atlantic Canada ───────────────────────────
# Monitoring sites, species observations, protected areas, watersheds

# Monitoring sites (water quality / biodiversity)
sites_data = {
    'site_id':   ['S01','S02','S03','S04','S05','S06','S07','S08'],
    'site_name': ['Petitcodiac River','Miramichi Lake','Fundy Shore',
                  'Tantramar Marsh','Kouchibouguac','Restigouche River',
                  'Saint John River','Annapolis Valley'],
    'province':  ['NB','NB','NB','NB','NB','NB','NB','NS'],
    'elevation_m':[12, 84, 5, 3, 8, 45, 18, 62],
    'geometry':  [
        Point(-64.96, 46.07), Point(-66.18, 46.85), Point(-65.07, 45.55),
        Point(-64.40, 45.90), Point(-64.95, 46.82), Point(-67.60, 47.70),
        Point(-66.64, 45.95), Point(-65.12, 45.05),
    ],
}
sites = gpd.GeoDataFrame(sites_data, crs='EPSG:4326')

# Protected areas (simplified polygons)
prot_data = {
    'pa_id':    ['PA01','PA02','PA03'],
    'name':     ['Fundy National Park','Kouchibouguac NP','Kejimkujik NP'],
    'type':     ['National Park','National Park','National Park'],
    'area_km2': [207, 239, 404],
    'geometry': [
        Polygon([(-65.4,45.4),(-64.6,45.4),(-64.6,45.7),(-65.4,45.7)]),
        Polygon([(-64.9,46.6),(-64.7,46.6),(-64.7,47.0),(-64.9,47.0)]),
        Polygon([(-65.4,44.5),(-64.8,44.5),(-64.8,44.9),(-65.4,44.9)]),
    ],
}
protected = gpd.GeoDataFrame(prot_data, crs='EPSG:4326')

# Watersheds
ws_data = {
    'ws_id':   ['WS01','WS02','WS03'],
    'name':    ['Saint John Watershed','Miramichi Watershed','Petitcodiac Watershed'],
    'area_km2':[55200, 13800, 3100],
    'geometry':[
        Polygon([(-68.0,46.8),(-66.0,46.8),(-66.0,45.0),(-68.0,45.0)]),
        Polygon([(-66.5,47.5),(-65.5,47.5),(-65.5,46.5),(-66.5,46.5)]),
        Polygon([(-65.2,46.3),(-64.5,46.3),(-64.5,45.7),(-65.2,45.7)]),
    ],
}
watersheds = gpd.GeoDataFrame(ws_data, crs='EPSG:4326')

# Species observations
obs_data = {
    'obs_id':   range(1, 11),
    'species':  ['Atlantic Salmon','Atlantic Salmon','Blanding\'s Turtle',
                 'Piping Plover','Piping Plover','Blanding\'s Turtle',
                 'Atlantic Salmon','Little Brown Bat','Little Brown Bat','Piping Plover'],
    'year':     [2022,2023,2022,2023,2022,2023,2021,2023,2022,2021],
    'count':    [12, 8, 3, 1, 2, 1, 15, 47, 52, 3],
    'geometry': [
        Point(-64.96,46.07), Point(-67.60,47.70), Point(-64.40,45.90),
        Point(-65.07,45.55), Point(-64.95,46.82), Point(-66.18,46.85),
        Point(-66.64,45.95), Point(-66.18,46.85), Point(-64.96,46.07),
        Point(-64.40,45.90),
    ],
}
observations = gpd.GeoDataFrame(obs_data, crs='EPSG:4326')

print("Ecological dataset loaded:")
for name, gdf in [('sites',sites),('protected_areas',protected),
                  ('watersheds',watersheds),('observations',observations)]:
    print(f"  {name:<18s}: {len(gdf)} features, CRS={gdf.crs.to_epsg()}")


import geopandas as gpd
import pandas as pd

print("=== Spatial JOIN: matching features by location ===")
print()

# Spatial join: which watershed does each site fall in?
sites_in_ws = gpd.sjoin(
    sites[['site_id','site_name','geometry']],
    watersheds[['ws_id','name','geometry']],
    how='left',
    predicate='within'
)
print("Spatial JOIN: sites → watersheds (predicate='within'):")
print(f"  {'site_id':<6s}  {'site_name':<24s}  {'watershed'}")
print("  " + "-"*56)
for _, row in sites_in_ws.iterrows():
    ws = row['name'] if pd.notna(row.get('name')) else 'outside all watersheds'
    print(f"  {row.site_id:<6s}  {row.site_name:<24s}  {ws}")

print()
# Count species observations per watershed
obs_in_ws = gpd.sjoin(
    observations[['obs_id','species','count','geometry']],
    watersheds[['ws_id','name','geometry']],
    how='left', predicate='within'
)
ws_summary = (obs_in_ws.groupby('name')
              .agg(n_obs=('obs_id','count'), total_count=('count','sum'))
              .reset_index())
print("Species observations per watershed:")
for _, row in ws_summary.iterrows():
    print(f"  {row['name']:<30s}  obs={row.n_obs}  total_count={row.total_count}")

print()
print("PostGIS spatial JOIN:")
print("""
  -- Sites within each watershed:
  SELECT s.site_id, s.site_name, w.name AS watershed
  FROM   monitoring_sites s
  JOIN   watersheds w ON ST_Within(s.geom, w.geom)
  ORDER  BY w.name, s.site_id;

  -- LEFT JOIN to keep unmatched sites (outside all watersheds):
  SELECT s.site_id, s.site_name, w.name AS watershed
  FROM   monitoring_sites s
  LEFT JOIN watersheds w ON ST_Within(s.geom, w.geom);

  -- Use ST_Intersects for polygons that may overlap boundaries:
  SELECT s.site_id, w.name AS watershed
  FROM   monitoring_sites s
  JOIN   watersheds w ON ST_Intersects(s.geom, w.geom);
""")


---
## Nearest neighbour analysis

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import pyproj

# ── Ecological dataset: Atlantic Canada ───────────────────────────
# Monitoring sites, species observations, protected areas, watersheds

# Monitoring sites (water quality / biodiversity)
sites_data = {
    'site_id':   ['S01','S02','S03','S04','S05','S06','S07','S08'],
    'site_name': ['Petitcodiac River','Miramichi Lake','Fundy Shore',
                  'Tantramar Marsh','Kouchibouguac','Restigouche River',
                  'Saint John River','Annapolis Valley'],
    'province':  ['NB','NB','NB','NB','NB','NB','NB','NS'],
    'elevation_m':[12, 84, 5, 3, 8, 45, 18, 62],
    'geometry':  [
        Point(-64.96, 46.07), Point(-66.18, 46.85), Point(-65.07, 45.55),
        Point(-64.40, 45.90), Point(-64.95, 46.82), Point(-67.60, 47.70),
        Point(-66.64, 45.95), Point(-65.12, 45.05),
    ],
}
sites = gpd.GeoDataFrame(sites_data, crs='EPSG:4326')

# Protected areas (simplified polygons)
prot_data = {
    'pa_id':    ['PA01','PA02','PA03'],
    'name':     ['Fundy National Park','Kouchibouguac NP','Kejimkujik NP'],
    'type':     ['National Park','National Park','National Park'],
    'area_km2': [207, 239, 404],
    'geometry': [
        Polygon([(-65.4,45.4),(-64.6,45.4),(-64.6,45.7),(-65.4,45.7)]),
        Polygon([(-64.9,46.6),(-64.7,46.6),(-64.7,47.0),(-64.9,47.0)]),
        Polygon([(-65.4,44.5),(-64.8,44.5),(-64.8,44.9),(-65.4,44.9)]),
    ],
}
protected = gpd.GeoDataFrame(prot_data, crs='EPSG:4326')

# Watersheds
ws_data = {
    'ws_id':   ['WS01','WS02','WS03'],
    'name':    ['Saint John Watershed','Miramichi Watershed','Petitcodiac Watershed'],
    'area_km2':[55200, 13800, 3100],
    'geometry':[
        Polygon([(-68.0,46.8),(-66.0,46.8),(-66.0,45.0),(-68.0,45.0)]),
        Polygon([(-66.5,47.5),(-65.5,47.5),(-65.5,46.5),(-66.5,46.5)]),
        Polygon([(-65.2,46.3),(-64.5,46.3),(-64.5,45.7),(-65.2,45.7)]),
    ],
}
watersheds = gpd.GeoDataFrame(ws_data, crs='EPSG:4326')

# Species observations
obs_data = {
    'obs_id':   range(1, 11),
    'species':  ['Atlantic Salmon','Atlantic Salmon','Blanding\'s Turtle',
                 'Piping Plover','Piping Plover','Blanding\'s Turtle',
                 'Atlantic Salmon','Little Brown Bat','Little Brown Bat','Piping Plover'],
    'year':     [2022,2023,2022,2023,2022,2023,2021,2023,2022,2021],
    'count':    [12, 8, 3, 1, 2, 1, 15, 47, 52, 3],
    'geometry': [
        Point(-64.96,46.07), Point(-67.60,47.70), Point(-64.40,45.90),
        Point(-65.07,45.55), Point(-64.95,46.82), Point(-66.18,46.85),
        Point(-66.64,45.95), Point(-66.18,46.85), Point(-64.96,46.07),
        Point(-64.40,45.90),
    ],
}
observations = gpd.GeoDataFrame(obs_data, crs='EPSG:4326')

print("Ecological dataset loaded:")
for name, gdf in [('sites',sites),('protected_areas',protected),
                  ('watersheds',watersheds),('observations',observations)]:
    print(f"  {name:<18s}: {len(gdf)} features, CRS={gdf.crs.to_epsg()}")


import numpy as np

print("=== Nearest neighbour: find closest feature ===")
print()

# Project to UTM for accurate distances
sites_utm = sites.to_crs("EPSG:32620")
obs_utm   = observations.to_crs("EPSG:32620")
prot_utm  = protected.to_crs("EPSG:32620")

# For each observation, find the nearest monitoring site
print("Nearest monitoring site to each species observation:")
print(f"  {'obs_id':<7s}  {'species':<22s}  {'nearest_site':<10s}  {'distance_km'}")
print("  " + "-"*62)
for _, obs_row in obs_utm.iterrows():
    distances = sites_utm.geometry.distance(obs_row.geometry)
    nearest_idx = distances.idxmin()
    nearest_site = sites_utm.loc[nearest_idx]
    dist_km = distances[nearest_idx] / 1000
    print(f"  {obs_row.obs_id:<7d}  {obs_row.species:<22s}  {nearest_site.site_id:<10s}  {dist_km:.1f} km")

print()
print("PostGIS nearest neighbour (KNN) query:")
print("""
  -- Fast KNN using <-> operator (requires GIST index):
  SELECT s.site_id, s.site_name,
         ST_Distance(
             ST_Transform(o.geom, 32620),
             ST_Transform(s.geom, 32620)
         ) / 1000 AS dist_km
  FROM   species_observations o
  CROSS JOIN LATERAL (
      SELECT site_id, site_name, geom
      FROM   monitoring_sites
      ORDER  BY geom <-> o.geom        -- <-> uses spatial index
      LIMIT  1
  ) s
  WHERE  o.obs_id = 5
  ORDER  BY dist_km;

  -- <->  : distance KNN operator (uses GIST index — fast)
  -- <#>  : bounding-box distance (even faster, approximate)
  -- LATERAL allows ORDER BY referencing outer query
""")


---
## Buffer analysis

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import pyproj

# ── Ecological dataset: Atlantic Canada ───────────────────────────
# Monitoring sites, species observations, protected areas, watersheds

# Monitoring sites (water quality / biodiversity)
sites_data = {
    'site_id':   ['S01','S02','S03','S04','S05','S06','S07','S08'],
    'site_name': ['Petitcodiac River','Miramichi Lake','Fundy Shore',
                  'Tantramar Marsh','Kouchibouguac','Restigouche River',
                  'Saint John River','Annapolis Valley'],
    'province':  ['NB','NB','NB','NB','NB','NB','NB','NS'],
    'elevation_m':[12, 84, 5, 3, 8, 45, 18, 62],
    'geometry':  [
        Point(-64.96, 46.07), Point(-66.18, 46.85), Point(-65.07, 45.55),
        Point(-64.40, 45.90), Point(-64.95, 46.82), Point(-67.60, 47.70),
        Point(-66.64, 45.95), Point(-65.12, 45.05),
    ],
}
sites = gpd.GeoDataFrame(sites_data, crs='EPSG:4326')

# Protected areas (simplified polygons)
prot_data = {
    'pa_id':    ['PA01','PA02','PA03'],
    'name':     ['Fundy National Park','Kouchibouguac NP','Kejimkujik NP'],
    'type':     ['National Park','National Park','National Park'],
    'area_km2': [207, 239, 404],
    'geometry': [
        Polygon([(-65.4,45.4),(-64.6,45.4),(-64.6,45.7),(-65.4,45.7)]),
        Polygon([(-64.9,46.6),(-64.7,46.6),(-64.7,47.0),(-64.9,47.0)]),
        Polygon([(-65.4,44.5),(-64.8,44.5),(-64.8,44.9),(-65.4,44.9)]),
    ],
}
protected = gpd.GeoDataFrame(prot_data, crs='EPSG:4326')

# Watersheds
ws_data = {
    'ws_id':   ['WS01','WS02','WS03'],
    'name':    ['Saint John Watershed','Miramichi Watershed','Petitcodiac Watershed'],
    'area_km2':[55200, 13800, 3100],
    'geometry':[
        Polygon([(-68.0,46.8),(-66.0,46.8),(-66.0,45.0),(-68.0,45.0)]),
        Polygon([(-66.5,47.5),(-65.5,47.5),(-65.5,46.5),(-66.5,46.5)]),
        Polygon([(-65.2,46.3),(-64.5,46.3),(-64.5,45.7),(-65.2,45.7)]),
    ],
}
watersheds = gpd.GeoDataFrame(ws_data, crs='EPSG:4326')

# Species observations
obs_data = {
    'obs_id':   range(1, 11),
    'species':  ['Atlantic Salmon','Atlantic Salmon','Blanding\'s Turtle',
                 'Piping Plover','Piping Plover','Blanding\'s Turtle',
                 'Atlantic Salmon','Little Brown Bat','Little Brown Bat','Piping Plover'],
    'year':     [2022,2023,2022,2023,2022,2023,2021,2023,2022,2021],
    'count':    [12, 8, 3, 1, 2, 1, 15, 47, 52, 3],
    'geometry': [
        Point(-64.96,46.07), Point(-67.60,47.70), Point(-64.40,45.90),
        Point(-65.07,45.55), Point(-64.95,46.82), Point(-66.18,46.85),
        Point(-66.64,45.95), Point(-66.18,46.85), Point(-64.96,46.07),
        Point(-64.40,45.90),
    ],
}
observations = gpd.GeoDataFrame(obs_data, crs='EPSG:4326')

print("Ecological dataset loaded:")
for name, gdf in [('sites',sites),('protected_areas',protected),
                  ('watersheds',watersheds),('observations',observations)]:
    print(f"  {name:<18s}: {len(gdf)} features, CRS={gdf.crs.to_epsg()}")


print("=== Buffer analysis: zone of influence ===")
print()

# Buffer monitoring sites by 10km, find overlapping observations
sites_utm    = sites.to_crs("EPSG:32620")
obs_utm      = observations.to_crs("EPSG:32620")
prot_utm     = protected.to_crs("EPSG:32620")

# 10km buffer around each site
sites_10km = sites_utm.copy()
sites_10km['geometry'] = sites_utm.geometry.buffer(10_000)   # 10,000 metres
sites_10km = sites_10km.to_crs("EPSG:4326")

obs_in_buffer = gpd.sjoin(
    obs_utm.to_crs("EPSG:4326"),
    sites_10km[['site_id','site_name','geometry']],
    how='left', predicate='within'
)
print("Species observations within 10 km of monitoring sites:")
summary = (obs_in_buffer.dropna(subset=['site_id'])
           .groupby(['site_id','site_name'])
           .agg(n_species=('species', lambda x: x.nunique()),
                total_obs=('obs_id','count'))
           .reset_index())
for _, row in summary.iterrows():
    print(f"  {row.site_id} ({row.site_name:<22s})  {row.n_species} species, {row.total_obs} observations in 10km")

print()
# Protected area: 5km buffer for habitat connectivity analysis
prot_buf = prot_utm.copy()
prot_buf['geometry'] = prot_utm.geometry.buffer(5_000)
print("5km buffer around protected areas (connectivity zone):")
for _, row in prot_buf.iterrows():
    area_km2 = row.geometry.area / 1e6
    print(f"  {row.pa_id} {row.name:<28s}  buffer area={area_km2:.0f} km²")

print()
print("PostGIS buffer operations:")
pg_buf = [
    ("ST_Buffer(geom, dist)",             "Buffer geometry by dist (in CRS units)"),
    ("ST_Buffer(ST_Transform(g,32620), 10000)", "10km buffer (project first for metres)"),
    ("ST_Buffer(geom, dist, 'side=left')","One-sided buffer on a line"),
    ("ST_Buffer(geom, dist, 8)",          "8-segment circle approximation (faster)"),
    ("ST_Difference(buf, original)",      "Donut: buffer minus original geometry"),
    ("ST_Union(ST_Buffer(g1,d), ST_Buffer(g2,d))", "Merge overlapping buffers"),
]
for sql, desc in pg_buf:
    print(f"  {sql:<52s}  {desc}")


---
## Common Pitfalls

**1. Spatial join without a spatial index — full cross join**
`JOIN watersheds w ON ST_Within(s.geom, w.geom)` with no GIST index on either geometry column performs a nested-loop join checking every site against every watershed. Always create GIST indexes before running spatial joins.

**2. Using ST_Distance in ORDER BY for nearest neighbour**
`ORDER BY ST_Distance(site.geom, target.geom) LIMIT 1` computes exact distance for all rows then sorts. Use the `<->` KNN operator in a `LATERAL` subquery to use the spatial index for nearest-neighbour queries.

**3. Buffer + join creating duplicate rows**
A 10km buffer around a site may overlap multiple watersheds. `JOIN watersheds ON ST_Intersects(buffer, geom)` then returns one row per overlapping watershed, not one row per site. Use `ST_DWithin` with `DISTINCT` or aggregate after the join if you want one result per site.

**4. Spatial join predicate mismatch between data and index**
GeoPandas `gpd.sjoin(..., predicate='intersects')` calls Shapely's `intersects()`. PostGIS `ST_Intersects` behaves the same. But `predicate='contains'` in GeoPandas means left contains right, while `ST_Contains(left, right)` in PostGIS means the same. Always verify predicate direction.

**5. Overlay operations creating sliver polygons**
When two polygon datasets have slightly misaligned boundaries (due to different source data), overlay operations create tiny 'sliver' polygons at the boundaries. Filter out slivers by area: `WHERE ST_Area(geom) > 1000` (1000 m²).

**6. Nearest neighbour returning the point itself**
When finding the nearest neighbour of points that exist in both tables (or the same table), `ORDER BY geom <-> target.geom LIMIT 1` returns the point itself (distance=0). Add `WHERE other.id != target.id` to exclude self-matches.


---
*sql_methods_library - Samantha McGarrigle*